In [34]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [35]:
df = pd.read_csv('insurance.csv')

In [36]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
3,22,109.4,1.55,3.34,True,Mumbai,student,Medium
39,51,100.6,1.68,11.99,True,Bangalore,unemployed,High
25,59,60.2,1.55,30.00,False,Mysore,government_job,Low
30,35,89.6,1.73,32.97,False,Delhi,business_owner,Low
62,34,72.8,1.83,35.67,False,Chennai,business_owner,Low


In [37]:
df_feat = df.copy()

In [38]:
df_feat['bmi'] = df_feat['weight']/(df_feat['height']**2)

In [39]:
def age_group(age):
  if age <25:
    return 'young'
  elif age<45:
    return 'adult'
  elif age<60:
    return 'middle_aged'
  else:
    return 'senior'

In [40]:
df_feat['age_group'] = df_feat['age'].apply(age_group)

In [41]:
def lifestylr_risk(row):
  if row['smoker'] and row['bmi'] > 30:
    return 'high'
  elif row['smoker'] and row['bmi'] > 27:
    return 'medium'
  else:
    return 'low'

In [42]:
df_feat['lifestyle_risk'] = df_feat.apply(lifestylr_risk, axis=1)

In [43]:
tier1cities = ['Mumbai', 'Delhi', 'Banglore', 'Chennai', 'Kolkata', 'Hydrabad', 'Pune']
tier2cities = ['Jaipur', 'Chandigarh', 'Indore', 'Kota', 'Lucknow']

In [44]:
def city_tier(city):
  if city in tier1cities:
    return 'tier1'
  elif city in tier2cities:
    return 'tier2'
  else:
    return 'tier3'

In [45]:
df_feat['city_tier'] = df_feat['city'].apply(city_tier)

In [46]:
df_feat.drop(columns=['age', 'height', 'smoker', 'weight', 'city'], inplace=True)

In [47]:
x = df_feat.drop(['insurance_premium_category'], axis= 1)
y = df_feat['insurance_premium_category']

In [48]:
x['occupation'].unique()

<StringArray>
[       'retired',     'freelancer',        'student', 'government_job',
 'business_owner',     'unemployed',    'private_job']
Length: 7, dtype: str

In [49]:
categorical_features = ['age_group', 'lifestyle_risk', 'city_tier', 'occupation']
numerical_features = ['bmi', 'income_lpa']

In [50]:
preprocessor = ColumnTransformer(transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numerical_features)
    ])

In [51]:
pipeline = Pipeline(steps = [('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=65))])

In [52]:
#train-test-split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=38,
    stratify=y
)

pipeline.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['High','Low','Medium']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['income_lpa','occupation','bmi','age_group','lifestyle_risk','city_tier']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remain

In [53]:
y_pred = pipeline.predict(x_test)
accuracy_score(y_test, y_pred)

0.45

In [54]:
#MODEL downloadd
import pickle
from pathlib import Path

pickle_model_path = Path.cwd() / "model.pkl"

with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)

print(f"Model saved at: {pickle_model_path}")

Model saved at: c:\Users\Yash Prajapati\OneDrive\Desktop\projects\fastapi&mlmodel\model.pkl


In [57]:
import sklearn
import numpy as np
import pandas as pd

print("Scikit-learn:", sklearn.__version__)
# print("NumPy:", np.__version__)
# print("Pandas:", pd.__version__)

Scikit-learn: 1.9.1


In [56]:
y_pred = pipeline.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.45

Classification Report:
              precision    recall  f1-score   support

        High       0.50      0.17      0.25         6
         Low       0.55      0.86      0.67         7
      Medium       0.29      0.29      0.29         7

    accuracy                           0.45        20
   macro avg       0.44      0.44      0.40        20
weighted avg       0.44      0.45      0.41        20

